# 04 - Final Model Evaluation

This notebook evaluates the trained model on the test set to get final performance metrics.

**Purpose:**
- Load the already-trained model
- Evaluate on the test set (completely unseen data)
- Generate test metrics (accuracy, F1, ROC-AUC)
- Save results for the frontend to display

**Prerequisites:**
- `03_model_training.ipynb` must have been run
- The following files must exist:
  - `data/models/optuna_study.pkl` (for model architecture)
  - `data/models/flight_delay_model.pth` (trained weights)
  - `data/processed/test_features.npy` and `test_labels.npy` (test data)

In [11]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from datetime import datetime

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Imports successful!
PyTorch version: 2.8.0+cu129
CUDA available: True


## 1. Define Model Architecture

This is the same flexible architecture used in training.

In [2]:
class FlexibleFlightDelayPredictor(nn.Module):
    """
    Flexible neural network that can have any number of hidden layers.
    Used for Optuna hyperparameter optimization.
    """
    def __init__(self, input_size, hidden_sizes, dropout_rates):
        super().__init__()
        
        layers = []
        prev_size = input_size
        
        # Build hidden layers dynamically
        for hidden_size, dropout in zip(hidden_sizes, dropout_rates):
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.BatchNorm1d(hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        # Output layer (single neuron for binary classification)
        layers.append(nn.Linear(prev_size, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        # No sigmoid here - we use BCEWithLogitsLoss
        return self.network(x)

print("✓ Model architecture defined")

✓ Model architecture defined


## 2. Load Best Model Configuration

Load the Optuna study to get the best hyperparameters.

In [6]:
# Manual architecture specification (from saved model inspection)
print("Model Architecture (from saved weights):")
print("="*60)

# Architecture extracted from flight_delay_model.pth
n_layers = 3
hidden_sizes = [448, 512, 512]
dropout_rates = [0.3, 0.3, 0.3]  # Standard dropout, exact values don't matter for evaluation

print(f"Hidden layers: {n_layers}")
print(f"Layer sizes: {hidden_sizes}")
print(f"Dropout rates: {dropout_rates}")
print("="*60)
print("✓ Architecture configured")

Model Architecture (from saved weights):
Hidden layers: 3
Layer sizes: [448, 512, 512]
Dropout rates: [0.3, 0.3, 0.3]
✓ Architecture configured


## 3. Load Trained Model Weights

In [7]:
# Initialize model with best architecture
input_size = 45  # Number of features
model = FlexibleFlightDelayPredictor(
    input_size=input_size,
    hidden_sizes=hidden_sizes,
    dropout_rates=dropout_rates
)

# Load trained weights
weights_path = Path('../data/models/flight_delay_model.pth')

if not weights_path.exists():
    raise FileNotFoundError(f"Model weights not found at {weights_path}. Run 03_model_training.ipynb first.")

model.load_state_dict(torch.load(weights_path))
model.eval()  # Set to evaluation mode

print("✓ Model weights loaded successfully")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

✓ Model weights loaded successfully
  Total parameters: 516,609


## 4. Load Test Dataset

In [12]:
# Load test data (from CSV)
test_features_path = Path('../data/processed/X_test.csv')
test_labels_path = Path('../data/processed/y_test.csv')

if not test_features_path.exists():
    raise FileNotFoundError(f"Test features not found at {test_features_path}. Run 02_feature_engineering.ipynb first.")

if not test_labels_path.exists():
    raise FileNotFoundError(f"Test labels not found at {test_labels_path}. Run 02_feature_engineering.ipynb first.")

# Load CSVs
X_test = pd.read_csv(test_features_path).values
y_test = pd.read_csv(test_labels_path).values.ravel()  # Flatten to 1D array

print(f"✓ Test data loaded")
print(f"  Test samples: {len(X_test):,}")
print(f"  Features: {X_test.shape[1]}")
print(f"  Delay rate: {y_test.mean()*100:.2f}%")

# Convert to PyTorch tensors
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

✓ Test data loaded
  Test samples: 1,207,883
  Features: 45
  Delay rate: 19.95%


## 5. Evaluate on Test Set

This is the **true** measure of model performance on completely unseen data.

In [ ]:
print("="*60)
print("EVALUATING ON TEST SET")
print("="*60)

# Make predictions
with torch.no_grad():
    # Get raw logits
    logits = model(X_test_tensor).squeeze()
    
    # Apply sigmoid to get probabilities
    probabilities = torch.sigmoid(logits)
    
    # Convert to binary predictions (threshold = 0.5)
    predictions = (probabilities > 0.5).float()

# Convert to numpy for sklearn metrics
y_test_np = y_test_tensor.numpy()
predictions_np = predictions.numpy()
probabilities_np = probabilities.numpy()

# Calculate metrics
test_accuracy = accuracy_score(y_test_np, predictions_np) * 100
test_f1 = f1_score(y_test_np, predictions_np)
test_roc_auc = roc_auc_score(y_test_np, probabilities_np)

# Calculate loss (using BCEWithLogitsLoss like in training)
criterion = nn.BCEWithLogitsLoss()
test_loss = criterion(logits, y_test_tensor).item()

print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test ROC-AUC:  {test_roc_auc:.4f}")
print("="*60)

EVALUATING ON TEST SET
Test Loss:     0.5435
Test Accuracy: 73.56%
Test F1 Score: 0.4734
Test ROC-AUC:  0.7516
⚠ F1 score is 0.4734 (goal was > 0.5)


## 6. Detailed Analysis

Confusion matrix and classification report.

In [14]:
# Confusion Matrix
cm = confusion_matrix(y_test_np, predictions_np)
tn, fp, fn, tp = cm.ravel()

print("\nConfusion Matrix:")
print("="*40)
print(f"True Negatives:  {tn:,} (correctly predicted on-time)")
print(f"False Positives: {fp:,} (predicted delay, was on-time)")
print(f"False Negatives: {fn:,} (predicted on-time, was delayed)")
print(f"True Positives:  {tp:,} (correctly predicted delay)")
print("="*40)

# Additional metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print(f"\nPrecision:   {precision:.4f} (of predicted delays, how many were correct)")
print(f"Recall:      {recall:.4f} (of actual delays, how many did we catch)")
print(f"Specificity: {specificity:.4f} (of actual on-time, how many did we catch)")

# Full classification report
print("\nClassification Report:")
print("="*60)
print(classification_report(y_test_np, predictions_np, 
                           target_names=['On-Time', 'Delayed'],
                           digits=4))


Confusion Matrix:
True Negatives:  745,023 (correctly predicted on-time)
False Positives: 221,864 (predicted delay, was on-time)
False Negatives: 97,480 (predicted on-time, was delayed)
True Positives:  143,516 (correctly predicted delay)

Precision:   0.3928 (of predicted delays, how many were correct)
Recall:      0.5955 (of actual delays, how many did we catch)
Specificity: 0.7705 (of actual on-time, how many did we catch)

Classification Report:
              precision    recall  f1-score   support

     On-Time     0.8843    0.7705    0.8235    966887
     Delayed     0.3928    0.5955    0.4734    240996

    accuracy                         0.7356   1207883
   macro avg     0.6385    0.6830    0.6484   1207883
weighted avg     0.7862    0.7356    0.7536   1207883



## 7. Save Test Metrics

Save to JSON so the API can serve them to the frontend.

In [15]:
# Create comprehensive test results
test_results = {
    "evaluated_at": datetime.now().isoformat(),
    "model_version": "2.0",
    "test_samples": int(len(X_test)),
    
    # Main metrics
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "test_f1": float(test_f1),
    "test_roc_auc": float(test_roc_auc),
    
    # Additional metrics
    "test_precision": float(precision),
    "test_recall": float(recall),
    "test_specificity": float(specificity),
    
    # Confusion matrix
    "confusion_matrix": {
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    },
    
    # Success criterion
    "success_criterion_met": bool(test_f1 > 0.5),
    "target_f1": 0.5
}

# Save to file
output_path = Path('../data/models/test_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"✓ Test results saved to {output_path}")

# Also save to API directory for serving
api_output_path = Path('../dotnet/FlightPredictor.API/Data/test_results.json')
api_output_path.parent.mkdir(parents=True, exist_ok=True)

with open(api_output_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f"✓ Test results copied to {api_output_path}")
print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)

✓ Test results saved to ..\data\models\test_results.json
✓ Test results copied to ..\dotnet\FlightPredictor.API\Data\test_results.json

EVALUATION COMPLETE


## Summary

This notebook evaluated your trained model on the test set - data that was **never** used during training or hyperparameter tuning.

**Key Points:**
- Test metrics show true generalization performance
- F1 score is the primary metric for this imbalanced classification problem
- Results are saved for the API to serve to your frontend

**Next Steps:**
1. Update your API to serve test results
2. Display test metrics in your frontend alongside validation metrics
3. Use test F1 score in your capstone presentation as the final performance metric